# Actividad 1: Temas Selectos II (Aprendizaje por Refuerzo)

Nuria Arroyo

**Fecha:** 29 de Marzo de 2026

Esta notebook implementa los ejercicios de la actividad 1, resolviendo los MDPs mediante Backward Induction y evaluando políticas propuestas.

In [19]:
import numpy as np
import random
import plotly.graph_objects as go


def evaluate_policy(pi, X, A, r, Q, N):
    # V[t][x] : value at time t for state x
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    # V[N] is already 0
    
    for t in range(N-1, -1, -1):  # from N-1 down to 0
        for x in X:
            a = pi[x]
            reward = r[(x, a)]
            next_value = sum(Q[(x, a)][i] * V[t+1][X[i]] for i in range(len(X)))
            V[t][x] = reward + next_value
    
    return V

def evaluate_randomized_policy(pi, X, A, r, Q, N):
    # pi is dict x -> dict a -> prob
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    
    for t in range(N-1, -1, -1):
        for x in X:
            value = 0.0
            for a in A[x]:
                prob_a = pi[x][a]
                reward = r[(x, a)]
                next_value = sum(Q[(x, a)][i] * V[t+1][X[i]] for i in range(len(X)))
                value += prob_a * (reward + next_value)
            V[t][x] = value
    
    return V

In [17]:
def evaluate_policy(pi, X, A, r, Q, N):
    # V[t][x] : value at time t for state x
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    # V[N] is already 0
    
    for t in range(N-1, -1, -1):  # from N-1 down to 0
        for x in X:
            a = pi[x]
            reward = r[(x, a)]
            next_value = sum(Q[(x, a)][i] * V[t+1][X[i]] for i in range(len(X)))
            V[t][x] = reward + next_value
    
    return V

def evaluate_randomized_policy(pi, X, A, r, Q, N):
    # pi is dict x -> dict a -> prob
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    
    for t in range(N-1, -1, -1):
        for x in X:
            value = 0.0
            for a in A[x]:
                prob_a = pi[x][a]
                reward = r[(x, a)]
                next_value = sum(Q[(x, a)][i] * V[t+1][X[i]] for i in range(len(X)))
                value += prob_a * (reward + next_value)
            V[t][x] = value
    
    return V

def simulate_trajectory(pi, X, A, r, Q, N, initial_state, num_trajectories=5000):
    total_rewards = []
    for _ in range(num_trajectories):
        x = initial_state
        reward_sum = 0
        for t in range(N):
            if isinstance(pi[x], dict):  # randomized
                actions = list(pi[x].keys())
                probs = list(pi[x].values())
                a = random.choices(actions, weights=probs)[0]
            else:  # deterministic
                a = pi[x]
            reward_sum += r[(x, a)]
            # next state
            probs_next = Q[(x, a)]
            x = random.choices(X, weights=probs_next)[0]
        total_rewards.append(reward_sum)
    return total_rewards

def plot_simulation_histogram(rewards, title="Histograma de Simulación Monte Carlo"):
    fig = go.Figure(data=[go.Histogram(x=rewards)])
    fig.update_layout(title=title, xaxis_title="Recompensa Total", yaxis_title="Frecuencia")
    fig.show()

def evaluate_policy_cost(pi, X, P_d, K, c_unit, h, p, N):
    """Evalúa un costo de política determinista usando Backward Induction."""
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    for t in range(N-1, -1, -1):
        for x in X:
            a = pi[x]
            ec = expected_cost_inventory(x, a, P_d, K, c_unit, h, p)
            next_value = sum(P_d[d] * V[t+1][max(0, x + a - d)] for d in range(len(P_d)))
            V[t][x] = ec + next_value
    return V

def evaluate_randomized_policy_cost(pi, X, P_d, K, c_unit, h, p, N):
    """Evalúa un costo de política aleatorizada usando Backward Induction."""
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    for t in range(N-1, -1, -1):
        for x in X:
            val = 0.0
            for a, prob in pi[x].items():
                ec = expected_cost_inventory(x, a, P_d, K, c_unit, h, p)
                next_value = sum(P_d[d] * V[t+1][max(0, x + a - d)] for d in range(len(P_d)))
                val += prob * (ec + next_value)
            V[t][x] = val
    return V

In [28]:
def evaluate_policy_cost(pi, X, P_d, K, c_unit, h, p, N):
    """Evalúa un costo de política determinista usando Backward Induction."""
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    for t in range(N-1, -1, -1):
        for x in X:
            a = pi[x]
            ec = expected_cost_inventory(x, a, P_d, K, c_unit, h, p)
            next_value = sum(P_d[d] * V[t+1][max(0, x + a - d)] for d in range(len(P_d)))
            V[t][x] = ec + next_value
    return V


def evaluate_randomized_policy_cost(pi, X, P_d, K, c_unit, h, p, N):
    """Evalúa un costo de política aleatorizada usando Backward Induction."""
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    for t in range(N-1, -1, -1):
        for x in X:
            val = 0.0
            for a, prob in pi[x].items():
                ec = expected_cost_inventory(x, a, P_d, K, c_unit, h, p)
                next_value = sum(P_d[d] * V[t+1][max(0, x + a - d)] for d in range(len(P_d)))
                val += prob * (ec + next_value)
            V[t][x] = val
    return V

In [21]:
def expected_cost_inventory(x, a, P_d, K, c_unit, h, p):
    """
    Calcula el costo esperado para una acción en un estado dado.
    E[c(x,a)] = K*(a>0) + c*a + E[h*X_{t+1} + p*shortage]
    """
    cost = K * (a > 0) + c_unit * a
    for d, prob in enumerate(P_d):
        inv_end = max(0, x + a - d)
        shortage = max(0, d - (x + a))
        cost += prob * (h * inv_end + p * shortage)
    return cost

def backward_induction_cost_minimization(X, A, P_d, K, c_unit, h, p, N):
    """
    Backward Induction para minimizar costo en MDP de inventario.
    """
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    policy = {t: {x: None for x in X} for t in range(N)}
    
    for t in range(N-1, -1, -1):
        for x in X:
            min_value = float('inf')
            best_a = None
            for a in A[x]:
                ec = expected_cost_inventory(x, a, P_d, K, c_unit, h, p)
                next_value = sum(P_d[d] * V[t+1][max(0, x + a - d)] for d in range(len(P_d)))
                value = ec + next_value
                if value < min_value:
                    min_value = value
                    best_a = a
            V[t][x] = min_value
            policy[t][x] = best_a
    
    return V, policy

## Ejercicio 1

Considere un MDP estacionario con espacio de estados:
X = {s1, s2, s3}.

Interpretación:
- s1: sistema en estado favorable,
- s2: sistema en estado intermedio,
- s3: sistema en estado crítico.

En cada estado se pueden tomar las siguientes acciones:
- A(s1) = {a1, a2}
- A(s2) = {a1, a2}
- A(s3) = {a2}

Las recompensas inmediatas están dadas por:
- r(s1, a1) = 6, r(s1, a2) = 4
- r(s2, a1) = 3, r(s2, a2) = 5
- r(s3, a2) = -2

Las probabilidades de transición están dadas por:
- P(·|s1, a1) = (0.7, 0.3, 0)
- P(·|s1, a2) = (0.4, 0.6, 0)
- P(·|s2, a1) = (0.2, 0.5, 0.3)
- P(·|s2, a2) = (0.1, 0.6, 0.3)
- P(·|s3, a2) = (0.0, 0.4, 0.6)

Se considera un horizonte finito N = 4.

1. Defina el MDP con los elementos anteriores.

In [11]:
# Definición del MDP
X = ['s1', 's2', 's3']  # Estados
A = {'s1': ['a1', 'a2'], 's2': ['a1', 'a2'], 's3': ['a2']}  # Acciones por estado
r = {('s1', 'a1'): 6, ('s1', 'a2'): 4, ('s2', 'a1'): 3, ('s2', 'a2'): 5, ('s3', 'a2'): -2}  # Recompensas
Q = {('s1', 'a1'): [0.7, 0.3, 0],
     ('s1', 'a2'): [0.4, 0.6, 0], 
     ('s2', 'a1'): [0.2, 0.5, 0.3],
     ('s2', 'a2'): [0.1, 0.6, 0.3], 
     ('s3', 'a2'): [0.0, 0.4, 0.6]}  # Probabilidades de transición
N = 4  # Horizonte finito

2. Ecuación de Bellman para horizonte finito

La ecuación de Bellman para un MDP con horizonte finito N es:

$$ V_t(x) = \max_{a \in A(x)} \left[ r(x, a) + \sum_{y \in X} Q(y | x, a) \cdot V_{t+1}(y) \right] $$

para t = 0, 1, ..., N-1, con V_N(x) = 0 para todo x.

Donde:
- $ V_t(x) $ es la función de valor en el tiempo t para el estado x.
- $ r(x, a) $ es la recompensa inmediata.
- $ Q(y | x, a) $ es la probabilidad de transición de x a y bajo acción a.

3. Evaluación de políticas Markovianas deterministas

El equipo debe elegir una de las siguientes alternativas de políticas deterministas:

**Alternativa 1:** π(s1) = a1, π(s2) = a1, π(s3) = a2

**Alternativa 2:** π(s1) = a2, π(s2) = a2, π(s3) = a2

**Alternativa 3:** π(s1) = a1, π(s2) = a2, π(s3) = a2


**Alternativa 2:** π(s1) = a2, π(s2) = a2, π(s3) = a2

**Alternativa 3:** π(s1) = a1, π(s2) = a2, π(s3) = a2

In [12]:

pi1 = {'s1': 'a1', 's2': 'a1', 's3': 'a2'}
V1 = evaluate_policy(pi1, X, A, r, Q, N)
print("Política 1 - Valores V_t(x):")
for t in range(N+1):
    print(f"t={t}: {V1[t]}")




Política 1 - Valores V_t(x):
t=0: {'s1': 18.860999999999997, 's2': 9.297, 's3': 0.1720000000000006}
t=1: {'s1': 15.299999999999999, 's2': 7.17, 's3': -1.1599999999999997}
t=2: {'s1': 11.1, 's2': 5.1, 's3': -1.9999999999999998}
t=3: {'s1': 6.0, 's2': 3.0, 's3': -2.0}
t=4: {'s1': 0.0, 's2': 0.0, 's3': 0.0}


 4. Evaluación de políticas Markovianas aleatorizadas

El equipo debe elegir una de las siguientes alternativas de políticas aleatorizadas:

**Random 1:** π(a1|s1) = 0.5, π(a2|s1) = 0.5; π(a1|s2) = 0.5, π(a2|s2) = 0.5; π(a2|s3) = 1.0


**Random 2:** π(a1|s1) = 0.7, π(a2|s1) = 0.3; π(a1|s2) = 0.3, π(a2|s2) = 0.7; π(a2|s3) = 1.0

**Random 3:** π(a1|s1) = 0.2, π(a2|s1) = 0.8; π(a1|s2) = 0.8, π(a2|s2) = 0.2; π(a2|s3) = 1.0

In [9]:
pi_rand1 = {'s1': {'a1': 0.5, 'a2': 0.5}, 's2': {'a1': 0.5, 'a2': 0.5}, 's3': {'a2': 1.0}}
V_rand1 = evaluate_randomized_policy(pi_rand1, X, A, r, Q, N)
print("Política aleatorizada 1 - Valores V_t(x):")
for t in range(N+1):
    print(f"t={t}: {V_rand1[t]}")

Política aleatorizada 1 - Valores V_t(x):
t=0: {'s1': 16.01075, 's2': 10.485249999999999, 's3': 1.1260000000000003}
t=1: {'s1': 13.11, 's2': 8.445, 's3': -0.41999999999999993}
t=2: {'s1': 9.55, 's2': 6.35, 's3': -1.5999999999999999}
t=3: {'s1': 5.0, 's2': 4.0, 's3': -2.0}
t=4: {'s1': 0.0, 's2': 0.0, 's3': 0.0}


5. Compare ambas políticas en términos de recompensa total esperada.

In [10]:
#comparar V1 y V_rand1 para cada estado y tiempo
print("\nComparación de políticas:")
for t in range(N+1):
    print(f"t={t}:")
    for x in X:
        print(f"  Estado {x}: V1={V1[t][x]:.2f}, V_rand1={V_rand1[t][x]:.2f}")


Comparación de políticas:
t=0:
  Estado s1: V1=18.86, V_rand1=16.01
  Estado s2: V1=9.30, V_rand1=10.49
  Estado s3: V1=0.17, V_rand1=1.13
t=1:
  Estado s1: V1=15.30, V_rand1=13.11
  Estado s2: V1=7.17, V_rand1=8.45
  Estado s3: V1=-1.16, V_rand1=-0.42
t=2:
  Estado s1: V1=11.10, V_rand1=9.55
  Estado s2: V1=5.10, V_rand1=6.35
  Estado s3: V1=-2.00, V_rand1=-1.60
t=3:
  Estado s1: V1=6.00, V_rand1=5.00
  Estado s2: V1=3.00, V_rand1=4.00
  Estado s3: V1=-2.00, V_rand1=-2.00
t=4:
  Estado s1: V1=0.00, V_rand1=0.00
  Estado s2: V1=0.00, V_rand1=0.00
  Estado s3: V1=0.00, V_rand1=0.00


In [20]:
# Simulación Monte Carlo para política 1
rewards = simulate_trajectory(pi1, X, A, r, Q, N, 's1')
mean_reward = np.mean(rewards)
std_reward = np.std(rewards)
print(f"Simulación Monte Carlo para política 1 desde s1: media={mean_reward:.2f}, std={std_reward:.2f}")
plot_simulation_histogram(rewards, "Histograma de Recompensas Totales - Política 1")

Simulación Monte Carlo para política 1 desde s1: media=18.91, std=5.42


In [15]:
# Ejemplo: simular para pi1 desde s1
rewards = simulate_trajectory(pi1, X, A, r, Q, N, 's1')
mean_reward = np.mean(rewards)
std_reward = np.std(rewards)
print(f"Simulación Monte Carlo para política 1 desde s1: media={mean_reward:.2f}, std={std_reward:.2f}")
plot_simulation_histogram(rewards, "Histograma de Recompensas Totales - Política 1")

Simulación Monte Carlo para política 1 desde s1: media=18.88, std=5.49


6. Verifique los resultados mediante simulación Monte Carlo (al menos
5000 trayectorias)

7. Interprete económicamente los resultados.

## Ejercicio 2

Considere un sistema de inventario con horizonte N=3.

**Estado al inicio del periodo:** $X_t \in \{0,1,2,3,4\}$

**Acción:** $a_t \in \{0,1,2\}$, sujeta a $X_t + a_t \leq 4$

**Demanda aleatoria:** $D_t \in \{0,1,2\}$, con 
$$P(D_t=0)=0.2, \quad P(D_t=1)=0.5, \quad P(D_t=2)=0.3$$

**Costos por periodo:**
- Costo fijo de ordenar: $K=2$ si $a_t > 0$
- Costo unitario: $c=1$
- Costo de mantener inventario: $h=1$ por unidad
- Penalización por faltante: $p=4$ por unidad no satisfecha

### 1. Especificación Formal del MDP $(X, A, Q, c)$

**Espacio de estados:** $X = \{0,1,2,3,4\}$ (niveles de inventario)

**Espacio de acciones:** $A(x) = \{0,1,2\}$ con restricción $x + a \leq 4$

**Función de costo por periodo:**
$$c(x,a) = K \cdot \mathbb{1}_{a>0} + c \cdot a + \mathbb{E}[h \cdot X_{t+1} + p \cdot S_t]$$

Donde $S_t = \max(0, D_t - (X_t + a_t))$ son unidades faltantes.

### 1. Especificación Formal del MDP $(X, A, Q, c)$

**Espacio de estados:** $X = \{0,1,2,3,4\}$ (niveles de inventario)

**Espacio de acciones:** $A(x) = \{0,1,2\}$ con restricción $x + a \leq 4$

**Función de costo por periodo:**
$$c(x,a) = K \cdot \mathbb{1}_{a>0} + c \cdot a + \mathbb{E}[h \cdot X_{t+1} + p \cdot S_t]$$

Donde $S_t = \max(0, D_t - (X_t + a_t))$ son unidades faltantes.

In [22]:
# Definición del MDP para Ejercicio 2
X2 = [0, 1, 2, 3, 4]
A2 = {x: list(range(5 - x)) for x in X2}  # a_t <= 4 - x
P_d = [0.2, 0.5, 0.3]  # P(D=0), P(D=1), P(D=2)
K = 2
c_unit = 1
h = 1
p = 4
N2 = 3


### 2. Derivación Paso a Paso de la Ley de Transición

**Modelo de dinámica:** Sistema de inventario con ventas perdidas.

**Paso 1: Inventario disponible después de ordenar**
$$I_t = X_t + a_t$$

**Paso 2: Demanda realizada**
$$D_t \sim \text{Discreto}(\{0,1,2\}), \quad P(D_t=0)=0.2, P(D_t=1)=0.5, P(D_t=2)=0.3$$

**Paso 3: Inventario final (modelo con ventas perdidas)**
$$X_{t+1} = \max\{0, X_t + a_t - D_t\}$$

Si $D_t > X_t + a_t$: hay falta de $S_t = D_t - (X_t + a_t)$ unidades

**Paso 4: Kernel de transición**

La probabilidad de transición es:
$$Q(x'|x,a) = P(X_{t+1} = x' | X_t = x, a_t = a)$$

Para cada demanda $d \in \{0,1,2\}$:
$$Q(x'|x,a) = \sum_{d=0}^{2} \mathbb{1}_{\max(0,x+a-d)=x'} \cdot P(D_t = d)$$

**Ejemplo:** Para $x=2, a=1$ (inventario $I=3$)
- Si $D_t=0$: $X_{t+1} = 3$
- Si $D_t=1$: $X_{t+1} = 2$
- Si $D_t=2$: $X_{t+1} = 1$

Entonces:
$$Q(3|2,1) = 0.2, \quad Q(2|2,1) = 0.5, \quad Q(1|2,1) = 0.3$$

3. Evalúe una política determinista tipo umbral propuesta por el equipo.



4. Evalúe una política Markoviana aleatorizada.


5. Compare el costo total esperado de ambas políticas.


6. Realice simulación Monte Carlo para validar resultados.

In [29]:
# 3,4,5,6 para Ejercicio 2: evaluación y comparación

# Política determinista tipo umbral propuesta del equipo (p.ej., S=2)
pi_det = {x: threshold_policy(x, 2) for x in X2}
V_det = evaluate_policy_cost(pi_det, X2, P_d, K, c_unit, h, p, N2)

# Política markoviana aleatorizada propuesta
pi_rand = {
    0: {0: 0.2, 1: 0.5, 2: 0.3},
    1: {0: 0.3, 1: 0.5, 2: 0.2},
    2: {0: 0.6, 1: 0.4},
    3: {0: 1.0},
    4: {0: 1.0}
}
V_rand = evaluate_randomized_policy_cost(pi_rand, X2, P_d, K, c_unit, h, p, N2)

print('Política determinista tipo umbral (S=2) - V_0:', V_det[0])
print('Política aleatorizada - V_0:', V_rand[0])

# Comparación costo total esperado
for x in X2:
    print(f"Estado x={x}: Det={V_det[0][x]:.2f}, Rand={V_rand[0][x]:.2f}")

# Monte Carlo para validar
costs_det = simulate_inventory_policy(pi_det, X2, P_d, K, c_unit, h, p, N2, initial_state=0)
costs_rand = []
for _ in range(5000):
    x = 0
    tot = 0
    for t in range(N2):
        a = random.choices(list(pi_rand[x].keys()), weights=list(pi_rand[x].values()))[0]
        tot += expected_cost_inventory(x, a, P_d, K, c_unit, h, p)
        d = random.choices([0,1,2], weights=P_d)[0]
        x = max(0, x + a - d)
    costs_rand.append(tot)

print(f"MC Det Umbral 2: mean={np.mean(costs_det):.2f}, std={np.std(costs_det):.2f}")
print(f"MC Rand: mean={np.mean(costs_rand):.2f}, std={np.std(costs_rand):.2f}")

plot_simulation_histogram(costs_det, 'MC Umbral 2: histograma de costos')
plot_simulation_histogram(costs_rand, 'MC Política aleatorizada: histograma de costos')

Política determinista tipo umbral (S=2) - V_0: {0: 12.100000000000001, 1: 11.1, 2: 8.1, 3: 7.180000000000001, 4: 7.140000000000001}
Política aleatorizada - V_0: {0: 12.72085, 1: 10.95891, 2: 9.312180000000001, 3: 7.8642, 4: 7.9620999999999995}
Estado x=0: Det=12.10, Rand=12.72
Estado x=1: Det=11.10, Rand=10.96
Estado x=2: Det=8.10, Rand=9.31
Estado x=3: Det=7.18, Rand=7.86
Estado x=4: Det=7.14, Rand=7.96
MC Det Umbral 2: mean=12.14, std=1.99
MC Rand: mean=12.70, std=1.59


## Ejercicio 3

Considere un MDP con espacio de estados:
X = {0,1,2}.

En cada periodo se elige una acción:
A(x) = {0,1},
donde la acción a=1 representa "intervenir" y a=0 "no intervenir".

Recompensa inmediata:
r(x,0) = x, r(x,1) = x - 2.

Probabilidades de transición:
P(·|x,0) = (0.6, 0.4, 0) si x=0,
          (0.0, 0.7, 0.3) si x=1,
          (0.0, 0.1, 0.9) si x=2.

P(·|x,1) = (1.0, 0.0, 0.0) si x=0,
          (0.0, 0.8, 0.2) si x=1,
          (0.0, 0.5, 0.5) si x=2.

Considere horizonte finito N=5.

### 2. Ecuación de Bellman para horizonte finito

$$V_t(x) = \max_{a \in A(x)} \left( r(x,a) + \sum_{y\in X} P(y|x,a)\,V_{t+1}(y) \right)$$

### 3. Resolución de etapa final $t = N-1$

En la etapa final, $V_N(x)=0$ para todo $x$ y para $t=N-1$:

$$V_{N-1}(x) = \max_{a \in A(x)} \left( r(x,a) + \sum_{y\in X} P(y|x,a)\cdot 0 \right) = \max_{a\in A(x)} r(x,a).$$

Cálculo explícito:
- Para $x=0$: acciones {0,1} --> $r(0,0)=0$, $r(0,1)=-2$ --> $V_{N-1}(0)=0$ con acción 0.
- Para $x=1$: acciones {0,1} --> $r(1,0)=1$, $r(1,1)=-1$ --> $V_{N-1}(1)=1$ con acción 0.
- Para $x=2$: acciones {0,1} --> $r(2,0)=2$, $r(2,1)=0$ --> $V_{N-1}(2)=2$ con acción 0.

### 4. Algoritmo de Backward Induction

1. Inicializar $V_N(x)=0$ para todo $x$. 
2. Para $t=N-1, N-2, ..., 0$:
   - Para cada $x\in X$:
     - Calcular $Q_a = r(x,a)+\sum_{y}P(y|x,a)V_{t+1}(y)$ para cada $a\in A(x)$.
     - Asignar $V_t(x)=\max_a Q_a$ y $\pi_t(x)=\arg\max_a Q_a$.
3. Devolver funciones $V_t(x)$ y política $\pi_t(x)$.

### 5. Implementación Python

(La implementación ya está en la celda de código de Ejercicio 3, usando `backward_induction_max`.)

### 6. Reporte de resultados
- Funciones de valor $V_t(x)$ impresas en consola.
- Política óptima por periodo impresa en consola.

### 7. Interpretación de la política óptima

- Estructura hallada: intervención (a=1) nunca es óptima en ningún tiempo porque r(x,0) >= r(x,1) + [beneficio esperado de cambio de estado] en todos los estados.
- Esto produce política conservadora: no intervenir (a=0) en todos los periodos.
- El resultado se explica porque intervenir reduce inmediatamente el reward en 2 y las mejoras de transición no compensan dentro del horizonte N=5.

In [13]:
# Definición del MDP para Ejercicio 3
X3 = [0, 1, 2]
A3 = {0: [0, 1], 1: [0, 1], 2: [0, 1]}
r3 = {(0, 0): 0, (0, 1): -2, (1, 0): 1, (1, 1): -1, (2, 0): 2, (2, 1): 0}
Q3 = {(0, 0): [0.6, 0.4, 0], (0, 1): [1.0, 0.0, 0.0],
      (1, 0): [0.0, 0.7, 0.3], (1, 1): [0.0, 0.8, 0.2],
      (2, 0): [0.0, 0.1, 0.9], (2, 1): [0.0, 0.5, 0.5]}
N3 = 5


Función de valor óptima V_t(x):
t=0: {0: 3.4720000000000004, 1: 7.0207999999999995, 2: 9.326400000000001}
t=1: {0: 2.208, 1: 5.368, 2: 7.5440000000000005}
t=2: {0: 1.16, 1: 3.7800000000000002, 2: 5.74}
t=3: {0: 0.4, 1: 2.3, 2: 3.9000000000000004}
t=4: {0: 0.0, 1: 1.0, 2: 2.0}
t=5: {0: 0.0, 1: 0.0, 2: 0.0}
Política óptima π_t(x):
t=0: {0: 0, 1: 0, 2: 0}
t=1: {0: 0, 1: 0, 2: 0}
t=2: {0: 0, 1: 0, 2: 0}
t=3: {0: 0, 1: 0, 2: 0}
t=4: {0: 0, 1: 0, 2: 0}


2. Ecuación de Bellman para horizonte finito.   



## Ejercicio 4

Considere un sistema de inventario con horizonte N=4.
Estado: X_t ∈ {0,1,2,3}.
Acciones: a_t ∈ {0,1,2}, X_t + a_t ≤ 3.
Demanda aleatoria: P(D_t=0)=0.3, P(D_t=1)=0.4, P(D_t=2)=0.3.
Dinámica: X_{t+1} = max{0, X_t + a_t - D_t}.
Costos por periodo: K=2 si a_t>0, c=1, h=1, p=5.

In [14]:
# Definición del MDP para Ejercicio 4
X4 = [0, 1, 2, 3]
A4 = {x: list(range(4 - x)) for x in X4}  # a <= 3 - x
P_d4 = [0.3, 0.4, 0.3]
K4 = 2
c4 = 1
h4 = 1
p4 = 5
N4 = 4

def expected_cost4(x, a, P_d, K, c, h, p):
    cost = K * (a > 0) + c * a
    for d, prob in enumerate(P_d):
        inv_end = max(0, x + a - d)
        shortage = max(0, d - (x + a))
        cost += prob * (h * inv_end + p * shortage)
    return cost

def backward_induction_min(X, A, P_d, K, c, h, p, N):
    V = {t: {x: 0.0 for x in X} for t in range(N+1)}
    policy = {t: {x: None for x in X} for t in range(N)}
    
    for t in range(N-1, -1, -1):
        for x in X:
            min_value = float('inf')
            best_a = None
            for a in A[x]:
                ec = expected_cost4(x, a, P_d, K, c, h, p)
                next_value = sum(P_d[d] * V[t+1][max(0, x + a - d)] for d in range(len(P_d)))
                value = ec + next_value
                if value < min_value:
                    min_value = value
                    best_a = a
            V[t][x] = min_value
            policy[t][x] = best_a
    
    return V, policy

V4, pi4_opt = backward_induction_min(X4, A4, P_d4, K4, c4, h4, p4, N4)

print("Función de valor óptima V_t(x):")
for t in range(N4+1):
    print(f"t={t}: {V4[t]}")

print("Política óptima π_t(x):")
for t in range(N4):
    print(f"t={t}: {pi4_opt[t]}")

Función de valor óptima V_t(x):
t=0: {0: 13.6264, 1: 11.808800000000002, 2: 9.6264, 3: 9.0888}
t=1: {0: 10.556000000000001, 1: 8.732000000000001, 2: 6.556, 3: 6.156000000000001}
t=2: {0: 7.46, 1: 5.7, 2: 3.46, 3: 3.54}
t=3: {0: 4.8, 1: 1.8, 2: 1.0, 3: 2.0}
t=4: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
Política óptima π_t(x):
t=0: {0: 2, 1: 0, 2: 0, 3: 0}
t=1: {0: 2, 1: 0, 2: 0, 3: 0}
t=2: {0: 2, 1: 0, 2: 0, 3: 0}
t=3: {0: 1, 1: 0, 2: 0, 3: 0}


### Análisis para Ejercicio 4

- La política óptima muestra una estructura tipo umbral: ordenar para alcanzar cierto nivel cuando el inventario es bajo.
- Comparando periodos inicial y final: en periodos iniciales, se ordena más para cubrir demanda futura, mientras que al final se minimiza orden.

El notebook está completo con implementaciones para todos los ejercicios. El equipo debe elegir alternativas para las evaluaciones de políticas, realizar simulaciones y proporcionar interpretaciones económicas en el video.